# 🕷️ Member 1 — Main Crawler (Phase 1)
## MYFutureJobs.gov.my — Categories: IT/Technology & Manufacturing/Engineering

**Target:** ~25,000 records from 2 sectors  
**Fields:** job title, job type, salary, location/state, employer, sector, posted date, job ID  
**Site:** `candidates.myfuturejobs.gov.my/search-jobs`  
**Method:** Selenium + BeautifulSoup (SPA requires JS rendering)  
**Output:** `raw_cat1_cat2.json` saved progressively to Google Drive

---

### ⚠️ Ethical & Legal Notes
- Always check `myfuturejobs.gov.my/robots.txt` before running  
- Apply **2-second delay** between requests (see `CRAWL_DELAY`)  
- This is a **public government portal** — only scrape publicly visible data  
- Do not attempt to log in or access protected data  
- For academic/research use only

## Step 0 — Check robots.txt First (Mandatory)

In [ ]:
import urllib.robotparser
import urllib.request

rp = urllib.robotparser.RobotFileParser()
rp.set_url('https://myfuturejobs.gov.my/robots.txt')
try:
    rp.read()
    print('robots.txt content:')
    with urllib.request.urlopen('https://myfuturejobs.gov.my/robots.txt') as f:
        print(f.read().decode())
except Exception as e:
    print(f'Could not fetch robots.txt: {e}')

# Check if crawling search page is allowed
test_url = 'https://candidates.myfuturejobs.gov.my/search-jobs'
allowed = rp.can_fetch('*', test_url)
print(f'\nCan crawl {test_url}: {allowed}')

robots.txt content:
# START YOAST BLOCK
# ---------------------------
User-agent: *
Disallow:

Sitemap: https://myfuturejobs.gov.my/sitemap_index.xml
# ---------------------------
# END YOAST BLOCK

Can crawl https://candidates.myfuturejobs.gov.my/search-jobs: True


## Step 1 — Install Dependencies

In [ ]:
# Run this cell once — installs all required packages in Colab
!pip install selenium beautifulsoup4 lxml webdriver-manager tqdm -q
!apt-get install -y chromium-browser chromium-chromedriver > /dev/null 2>&1
print('✅ All dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 8.5 MB/s eta 0:00:00
✅ All dependencies installed


## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Change this to your shared group folder path ──
DRIVE_FOLDER = '/content/drive/MyDrive/HPDP_Project/data'
os.makedirs(DRIVE_FOLDER, exist_ok=True)

OUTPUT_FILE = os.path.join(DRIVE_FOLDER, 'raw_cat1_cat2.json')
print(f'✅ Drive mounted. Saving to: {OUTPUT_FILE}')

MessageError: Error: credential propagation was unsuccessful

## Step 3 — Configuration

In [ ]:
# ============================================================
#  CONFIGURATION — Adjust if needed
# ============================================================

CRAWL_DELAY     = 2       # seconds between page loads (ethical)
PAGE_SIZE       = 20      # listings per page on the site
MAX_PAGES       = 700     # safety cap: 700 pages x 20 = 14,000 per sector
TARGET_PER_CAT  = 12500   # aim for ~12,500 per category = 25,000 total
SAVE_EVERY      = 50      # save to Drive every N records
TIMEOUT         = 15      # Selenium wait timeout (seconds)
USER_AGENT      = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) '
    'Chrome/120.0.0.0 Safari/537.36'
)

# ── Sector filter values used in the MYFutureJobs search URL ──
# These are the two categories Member 1 owns.
# Inspect the site's network tab to confirm exact filter param names.
# Based on ESCO taxonomy used by MYFutureJobs:
CATEGORIES = {
    'cat1_IT_Technology': {
        'sector': 'Information and Communication Technology',
        'url_keyword': 'IT',
        'description': 'Software, IT, Cybersecurity, Data roles'
    },
    'cat2_Manufacturing_Engineering': {
        'sector': 'Manufacturing',
        'url_keyword': 'Engineering',
        'description': 'Manufacturing, Engineering, Technical roles'
    }
}

BASE_URL = 'https://candidates.myfuturejobs.gov.my/search-jobs'

print('✅ Configuration set')
print(f'   Targeting {TARGET_PER_CAT:,} records per category')
print(f'   Total target: {TARGET_PER_CAT * 2:,} records')
print(f'   Crawl delay: {CRAWL_DELAY}s')

## Step 4 — Helper Functions

In [ ]:
import time
import json
import random
import logging
from datetime import datetime
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger(__name__)


# ── Selenium driver setup (headless Chrome for Colab) ─────────
def create_driver():
    from selenium.webdriver.chrome.service import Service
    import shutil, os

    opts = Options()
    opts.add_argument('--headless=new')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument(f'--user-agent={USER_AGENT}')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])

    # Auto-detect chromedriver location (works in Colab & local)
    chromedriver_path = shutil.which('chromedriver') or '/usr/bin/chromedriver'
    chrome_path = shutil.which('chromium-browser') or shutil.which('chromium') or shutil.which('google-chrome')
    if chrome_path:
        opts.binary_location = chrome_path

    service = Service(executable_path=chromedriver_path)
    driver = webdriver.Chrome(service=service, options=opts)

    # Mask webdriver detection
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


# ── Wait for job cards to appear ──────────────────────────────
def wait_for_results(driver, timeout=TIMEOUT):
    """Wait until job listing cards are present in DOM."""
    wait = WebDriverWait(driver, timeout)
    try:
        # Adjust selector after inspecting the actual site DOM
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, 'div.job-card, div[class*="vacancy"], div[class*="job-item"], article')
        ))
        time.sleep(1.5)  # extra buffer for lazy-loaded content
        return True
    except TimeoutException:
        log.warning('Timed out waiting for job cards')
        return False


# ── Parse one job card from its HTML ─────────────────────────
def parse_job_card(card_html, category_name, page_num):
    """
    Extract fields from a single job card BeautifulSoup tag.
    IMPORTANT: Selector strings below are PLACEHOLDERS.
    After running Step 6 (DOM inspection), update them with real selectors.
    """
    soup = card_html  # already a BS4 tag

    def safe_text(selector, attr=None):
        """Return stripped text or attr value, or empty string on failure."""
        el = soup.select_one(selector)
        if el is None:
            return ''
        if attr:
            return el.get(attr, '').strip()
        return el.get_text(strip=True)

    # ── UPDATE SELECTORS AFTER DOM INSPECTION ──
    # Run Step 6 first to print the real HTML structure
    record = {
        'job_id':       safe_text('[data-job-id], [data-id], [id]', 'data-job-id')
                        or safe_text('[href]', 'href').split('jobId=')[-1].split('&')[0],
        'job_title':    safe_text('h2, h3, .job-title, [class*="title"], [class*="position"]'),
        'employer':     safe_text('.company-name, .employer, [class*="company"], [class*="employer"]'),
        'location':     safe_text('.location, [class*="location"], [class*="state"], [class*="region"]'),
        'salary':       safe_text('.salary, [class*="salary"], [class*="wage"], [class*="pay"]'),
        'job_type':     safe_text('.job-type, [class*="employment"], [class*="type"]'),
        'sector':       safe_text('.sector, .industry, [class*="sector"], [class*="industry"]'),
        'posted_date':  safe_text('.date, .posted, [class*="date"], [class*="posted"], time'),
        'listing_url':  safe_text('a[href*="jobId"], a[href*="search-jobs"]', 'href'),
        # Metadata
        'category':     category_name,
        'page_number':  page_num,
        'scraped_at':   datetime.utcnow().isoformat()
    }

    # Normalise listing URL to absolute
    if record['listing_url'] and not record['listing_url'].startswith('http'):
        record['listing_url'] = 'https://candidates.myfuturejobs.gov.my' + record['listing_url']

    return record


# ── Progressive save to Drive ─────────────────────────────────
def save_progress(records, filepath):
    """Append records to JSON file (creates if not exists)."""
    existing = []
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                existing = json.load(f)
        except json.JSONDecodeError:
            existing = []
    existing.extend(records)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)
    return len(existing)


# ── Get already-scraped job IDs (for dedup / resume) ─────────
def load_existing_ids(filepath):
    if not os.path.exists(filepath):
        return set()
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return {r.get('job_id', '') for r in data if r.get('job_id')}
    except Exception:
        return set()


print('✅ Helper functions defined')

## Step 5 — Build Search URLs

MYFutureJobs uses a React SPA at `candidates.myfuturejobs.gov.my/search-jobs`.  
Filters are applied via URL query parameters **and** via on-page filter controls.  
Run Step 6 first to discover the exact parameter names from a real page.

Common observed pattern:
```
https://candidates.myfuturejobs.gov.my/search-jobs?sector=<SECTOR>&page=<N>
```

In [ ]:
import urllib.parse

def build_search_url(sector_name, page=1):
    """
    Build paginated search URL for a given sector.
    Parameter names are based on observed MYFutureJobs URL structure.
    Update `sector_param` key after DOM inspection (Step 6).
    """
    params = {
        'sector': sector_name,   # ← verify exact param name in Step 6
        'page': page,
        'pageSize': PAGE_SIZE,
    }
    return f"{BASE_URL}?{urllib.parse.urlencode(params)}"


# Preview URLs
for cat_key, cat_info in CATEGORIES.items():
    url = build_search_url(cat_info['sector'], page=1)
    print(f'{cat_key}:')
    print(f'  {url}\n')

## Step 6 — DOM Inspection (Run This Before Full Crawl!)

This cell opens the site, prints the raw HTML, and helps you discover:
- The **CSS selectors** for job cards
- The **exact URL parameters** used for sector/page filters
- Whether pagination is URL-based or click-based

**Read the output carefully and update Step 4's `parse_job_card()` selectors.**

In [ ]:
driver = create_driver()

try:
    # Load the general search page first (no filter)
    driver.get(BASE_URL)
    print(f'Loaded: {driver.current_url}')
    time.sleep(4)  # let JS render

    # ── Print page title & final URL ──
    print(f'Title: {driver.title}')
    print(f'Final URL: {driver.current_url}')
    print()

    # ── Print first 5000 chars of rendered HTML ──
    soup = BeautifulSoup(driver.page_source, 'lxml')
    print('=== RENDERED HTML (first 5000 chars) ===')
    print(driver.page_source[:5000])
    print()

    # ── Try to find job card containers ──
    print('=== CANDIDATE CARD SELECTORS ===')
    for selector in [
        'div.job-card', 'div[class*="vacancy"]', 'div[class*="job-item"]',
        'div[class*="listing"]', 'article', 'li[class*="job"]',
        '.search-result', 'div[class*="card"]'
    ]:
        found = soup.select(selector)
        if found:
            print(f'  [{len(found)} cards] selector: {selector!r}')
            print(f'  First card snippet: {str(found[0])[:400]}')
            print()

    # ── Check for pagination controls ──
    print('=== PAGINATION ===')
    for sel in ['[class*="pagination"]', '[class*="page"]', 'button[aria-label*="Next"]', 'a[aria-label*="next"]']:
        el = soup.select(sel)
        if el:
            print(f'  Found: {sel!r} → {str(el[0])[:200]}')

    # ── Check network requests (XHR) made by the page ──
    # Intercept API calls to find direct JSON endpoints
    logs = driver.get_log('performance') if 'performance' in driver.log_types else []
    api_calls = [l for l in logs if 'api' in str(l).lower() or 'search' in str(l).lower()]
    if api_calls:
        print('\n=== DETECTED API CALLS ===')
        for l in api_calls[:10]:
            print(l)

finally:
    driver.quit()
    print('\n✅ Inspection complete — update selectors in Step 4 before crawling')

## Step 7 — (Optional) Check for Hidden JSON API

Many SPAs call a backend API that returns JSON directly.  
If found, we can skip Selenium entirely and use `requests` for much faster crawling.  
This cell tries common API endpoint patterns.

In [ ]:
import requests

HEADERS = {
    'User-Agent': USER_AGENT,
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://candidates.myfuturejobs.gov.my/search-jobs',
    'Origin': 'https://candidates.myfuturejobs.gov.my',
}

# Common API endpoint patterns to probe
CANDIDATE_ENDPOINTS = [
    'https://candidates.myfuturejobs.gov.my/api/vacancies',
    'https://candidates.myfuturejobs.gov.my/api/jobs',
    'https://candidates.myfuturejobs.gov.my/api/search',
    'https://candidates.myfuturejobs.gov.my/api/v1/jobs',
    'https://api.myfuturejobs.gov.my/api/vacancies',
    'https://api.myfuturejobs.gov.my/v1/jobs/search',
]

session = requests.Session()
session.headers.update(HEADERS)

print('Probing for JSON API endpoints...')
for url in CANDIDATE_ENDPOINTS:
    try:
        r = session.get(url, timeout=8, params={'page': 1, 'pageSize': 5})
        print(f'{r.status_code} — {url}')
        if r.status_code == 200:
            try:
                data = r.json()
                print(f'  ✅ JSON FOUND! Keys: {list(data.keys())[:8]}')
                print(f'  Preview: {str(data)[:500]}')
            except Exception:
                print(f'  Response (not JSON): {r.text[:200]}')
        time.sleep(1)
    except Exception as e:
        print(f'ERROR — {url}: {e}')

print('\n💡 If a JSON endpoint is found, update Step 9 to use requests instead of Selenium.')

## Step 8 — Determine Total Pages Available

Before the full crawl, count how many pages exist per sector  
so we can set realistic targets.

In [ ]:
def count_total_pages(driver, sector_name):
    """
    Load page 1 of a sector and extract total result count / page count.
    Update selectors after DOM inspection.
    """
    url = build_search_url(sector_name, page=1)
    driver.get(url)
    time.sleep(4)

    soup = BeautifulSoup(driver.page_source, 'lxml')

    # Try to find total result count text e.g. "Showing 1-20 of 15,432 jobs"
    total_text = ''
    for selector in [
        '[class*="total"]', '[class*="result-count"]',
        '[class*="count"]', 'h1', 'h2', 'span[class*="jobs"]'
    ]:
        el = soup.select_one(selector)
        if el and any(c.isdigit() for c in el.get_text()):
            total_text = el.get_text(strip=True)
            break

    print(f'Sector: {sector_name}')
    print(f'Total text found: {total_text!r}')

    # Try to extract number from text
    import re
    nums = re.findall(r'[\d,]+', total_text.replace(',', ''))
    total_jobs = int(nums[-1]) if nums else 0
    total_pages = (total_jobs // PAGE_SIZE) + 1 if total_jobs else MAX_PAGES

    print(f'Estimated total jobs: {total_jobs:,}')
    print(f'Estimated total pages: {total_pages:,}')
    return min(total_pages, MAX_PAGES)


driver = create_driver()
try:
    for cat_key, cat_info in CATEGORIES.items():
        pages = count_total_pages(driver, cat_info['sector'])
        CATEGORIES[cat_key]['total_pages'] = pages
        print(f'→ Will crawl up to {pages} pages\n')
finally:
    driver.quit()

## Step 9 — Main Crawler (Selenium-Based)

Crawls **Category 1 (IT/Technology)** and **Category 2 (Manufacturing/Engineering)**  
with pagination, error recovery, deduplication, and progressive saving.

In [ ]:
def crawl_category(category_key, category_info, output_file, existing_ids):
    """
    Full paginated crawl for one category sector.
    Returns list of new records collected.
    """
    sector      = category_info['sector']
    total_pages = category_info.get('total_pages', MAX_PAGES)
    new_records = []
    buffer      = []      # accumulate before saving
    consecutive_empty = 0

    log.info(f'Starting crawl: {category_key} | sector={sector} | pages≈{total_pages}')

    driver = create_driver()

    try:
        for page in tqdm(range(1, total_pages + 1), desc=category_key):

            if len(new_records) >= TARGET_PER_CAT:
                log.info(f'Reached target {TARGET_PER_CAT:,} records for {category_key}. Stopping.')
                break

            url = build_search_url(sector, page=page)

            # ── Load page with retry ──
            for attempt in range(3):
                try:
                    driver.get(url)
                    time.sleep(CRAWL_DELAY + random.uniform(0.3, 0.8))
                    success = wait_for_results(driver)
                    if success:
                        break
                    log.warning(f'Page {page} attempt {attempt+1}: no results yet')
                    time.sleep(3)
                except Exception as e:
                    log.error(f'Page {page} attempt {attempt+1} error: {e}')
                    time.sleep(5)
                    # Restart driver if crashed
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = create_driver()
            else:
                log.error(f'Skipping page {page} after 3 failed attempts')
                consecutive_empty += 1
                if consecutive_empty >= 5:
                    log.error('5 consecutive failures — stopping crawl')
                    break
                continue

            # ── Parse job cards ──
            soup = BeautifulSoup(driver.page_source, 'lxml')

            # UPDATE THIS SELECTOR after Step 6 inspection
            cards = soup.select('div.job-card, div[class*="vacancy-item"], div[class*="job-listing"], article')

            if not cards:
                log.warning(f'Page {page}: No cards found. HTML snippet: {str(soup)[:500]}')
                consecutive_empty += 1
                if consecutive_empty >= 5:
                    log.error('5 empty pages — likely end of results')
                    break
                continue

            consecutive_empty = 0
            page_new = 0

            for card in cards:
                record = parse_job_card(card, category_key, page)

                # Skip duplicates
                job_id = record.get('job_id', '')
                if job_id and job_id in existing_ids:
                    continue
                if job_id:
                    existing_ids.add(job_id)

                new_records.append(record)
                buffer.append(record)
                page_new += 1

            log.info(f'Page {page}/{total_pages} | +{page_new} new | total so far: {len(new_records):,}')

            # ── Progressive save ──
            if len(buffer) >= SAVE_EVERY:
                total_saved = save_progress(buffer, output_file)
                log.info(f'💾 Saved — total in file: {total_saved:,}')
                buffer = []

    finally:
        # Save any remaining buffer
        if buffer:
            save_progress(buffer, output_file)
        try:
            driver.quit()
        except Exception:
            pass

    log.info(f'✅ Finished {category_key}: {len(new_records):,} new records')
    return new_records


print('✅ Crawl function defined')

## Step 10 — RUN CRAWL (Category 1: IT/Technology)

In [ ]:
# Load existing IDs to support resume if the run was interrupted
existing_ids = load_existing_ids(OUTPUT_FILE)
print(f'Resuming from {len(existing_ids):,} already-scraped IDs')

# ── CRAWL CATEGORY 1 ──
cat1_records = crawl_category(
    category_key='cat1_IT_Technology',
    category_info=CATEGORIES['cat1_IT_Technology'],
    output_file=OUTPUT_FILE,
    existing_ids=existing_ids
)

print(f'\n📊 Category 1 complete: {len(cat1_records):,} records')

## Step 11 — RUN CRAWL (Category 2: Manufacturing/Engineering)

In [ ]:
# ── CRAWL CATEGORY 2 ──
cat2_records = crawl_category(
    category_key='cat2_Manufacturing_Engineering',
    category_info=CATEGORIES['cat2_Manufacturing_Engineering'],
    output_file=OUTPUT_FILE,
    existing_ids=existing_ids
)

print(f'\n📊 Category 2 complete: {len(cat2_records):,} records')
print(f'📊 Grand total (Cat 1 + Cat 2): {len(cat1_records) + len(cat2_records):,} records')

## Step 12 — Validate Output

In [ ]:
import pandas as pd

# Load saved file
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    all_records = json.load(f)

df = pd.DataFrame(all_records)

print('=' * 55)
print('VALIDATION REPORT — Member 1 Output')
print('=' * 55)
print(f'Total records         : {len(df):,}')
print(f'Unique job IDs        : {df["job_id"].nunique():,}')
print(f'Duplicate records     : {len(df) - df["job_id"].nunique():,}')
print()
print('Records per category:')
print(df['category'].value_counts().to_string())
print()
print('Missing values per field:')
print(df.replace('', pd.NA).isna().sum().to_string())
print()
print('Sample records:')
display(df.head(5))

# ── Milestone check ──
target = TARGET_PER_CAT * 2
pct = len(df) / target * 100
print(f'\n🎯 Target: {target:,} | Collected: {len(df):,} ({pct:.1f}%)')
if len(df) >= target:
    print('✅ TARGET MET — hand off raw_cat1_cat2.json to Member 2 for merging')
else:
    print(f'⚠️  Still need {target - len(df):,} more records')
    print('   → Try adding more sectors (see Step 13) or notify the group')

## Step 13 — (Fallback) Add More Sectors if Target Not Met

MYFutureJobs lists ~500,000 active vacancies annually.  
If IT + Manufacturing don't reach 25,000, add these supplementary sectors:

In [ ]:
# Run ONLY if Step 12 shows you are still below target

SUPPLEMENTARY_SECTORS = [
    'Finance and Insurance',
    'Retail Trade',
    'Healthcare and Social Services',
    'Education',
    'Construction',
]

# Determine how many more are needed
still_needed = max(0, (TARGET_PER_CAT * 2) - len(df))
print(f'Still need: {still_needed:,} more records')

if still_needed > 0:
    per_supp = still_needed // len(SUPPLEMENTARY_SECTORS) + 500  # buffer
    for sector in SUPPLEMENTARY_SECTORS:
        if still_needed <= 0:
            break
        supp_info = {
            'sector': sector,
            'total_pages': min(MAX_PAGES, (per_supp // PAGE_SIZE) + 5)
        }
        extra = crawl_category(
            category_key=f'supplementary_{sector.replace(" ", "_")}',
            category_info=supp_info,
            output_file=OUTPUT_FILE,
            existing_ids=existing_ids
        )
        still_needed -= len(extra)
        print(f'+{len(extra):,} from {sector} | Remaining needed: {still_needed:,}')
else:
    print('✅ Target already met — no supplementary crawling needed')

## Step 14 — Architecture Diagram (Member 1 Report Section)

Copy this Mermaid diagram into your report's Introduction / System Architecture section.

In [ ]:
architecture_diagram = """
```mermaid
flowchart TD
    A[myfuturejobs.gov.my\n candidates portal] -->|HTTP GET via Selenium| B[Headless Chrome\nDriver]
    B -->|Rendered HTML| C[BeautifulSoup Parser]
    C -->|Extracted Fields| D[Record Buffer]
    D -->|Every 50 records| E[(raw_cat1_cat2.json\nGoogle Drive)]

    subgraph Member1 [Member 1 — Crawler Phase 1]
        B
        C
        D
    end

    E -->|Merge| F[Member 2:\nraw_data.json\n100,000 records]
    F --> G[Member 3:\nclean_data.csv]
    G --> H[Member 4:\nOptimization + Report]

    style Member1 fill:#1a237e,color:#fff
```
"""
print(architecture_diagram)

## Step 15 — Review Member 2's Output (Cross-check Duty)

After Member 2 finishes crawling Categories 3 & 4, run this cell to validate their output.

In [ ]:
# Point to Member 2's file — update path
MEMBER2_FILE = os.path.join(DRIVE_FOLDER, 'raw_cat3_cat4.json')

if os.path.exists(MEMBER2_FILE):
    with open(MEMBER2_FILE, 'r', encoding='utf-8') as f:
        m2_data = json.load(f)
    m2_df = pd.DataFrame(m2_data)

    print('=== Member 2 Output Review ===')
    print(f'Total records: {len(m2_df):,}')
    print(f'Unique job IDs: {m2_df["job_id"].nunique():,}')
    print(f'Categories: {m2_df["category"].unique().tolist()}')
    print()
    print('Missing values:')
    print(m2_df.replace('', pd.NA).isna().sum().to_string())

    # Cross-duplicate check
    my_ids = set(df['job_id'].dropna())
    m2_ids = set(m2_df['job_id'].dropna())
    overlap = my_ids & m2_ids
    print(f'\nOverlapping job IDs with Member 1: {len(overlap):,}')
    if overlap:
        print('⚠️  Overlap found — Member 2 should deduplicate before merging')
    else:
        print('✅ No overlap — datasets are clean')
else:
    print(f'Member 2 file not found yet at: {MEMBER2_FILE}')

---
## 📋 Member 1 Checklist

| Task | Status |
|------|--------|
| Check robots.txt (Step 0) | ☐ |
| Install dependencies (Step 1) | ☐ |
| Mount Google Drive (Step 2) | ☐ |
| Configure output paths (Step 3) | ☐ |
| Run DOM inspection (Step 6) | ☐ |
| Update CSS selectors in `parse_job_card()` | ☐ |
| Check for hidden JSON API (Step 7) | ☐ |
| Count available pages (Step 8) | ☐ |
| Run Category 1 crawl (Step 10) | ☐ |
| Run Category 2 crawl (Step 11) | ☐ |
| Validate output ≥ 25,000 records (Step 12) | ☐ |
| Review Member 2's output (Step 15) | ☐ |
| Write Introduction + Architecture section for report | ☐ |
| Prepare Slides 1–4 | ☐ |